Grad-CAM++ implementation

In [ ]:
 
def make_gradcampp_heatmap(img_array, model, last_conv_layer_name, img_size=224):
    """
    img_array: (1, H, W, 3) preprocessed (after densenet_preprocess)
    """
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape(persistent=True) as tape1:
        with tf.GradientTape(persistent=True) as tape2:
            with tf.GradientTape() as tape3:
                conv_outputs, predictions = grad_model(img_array)
                loss = predictions[:, 0]     # single output neuron

            first_derivative = tape3.gradient(loss, conv_outputs)
        second_derivative = tape2.gradient(first_derivative, conv_outputs)
    third_derivative = tape1.gradient(second_derivative, conv_outputs)

    global_sum = tf.reduce_sum(conv_outputs, axis=(1, 2), keepdims=True)

    alpha_num = second_derivative
    alpha_den = 2.0 * second_derivative + third_derivative * global_sum
    alpha_den = tf.where(alpha_den != 0.0, alpha_den, tf.ones_like(alpha_den))
    alphas = alpha_num / (alpha_den + 1e-7)

    weights = tf.nn.relu(first_derivative)
    alphas_thresholding = tf.nn.relu(loss * weights)
    alphas /= tf.reduce_sum(alphas_thresholding, axis=(1, 2), keepdims=True) + 1e-7

    deep_linearization_weights = tf.reduce_sum(alphas * weights, axis=(1, 2))  # (1, C)

    conv_outputs = conv_outputs[0]                 # (H', W', C)
    deep_linearization_weights = deep_linearization_weights[0]  # (C,)

    cam = tf.reduce_sum(deep_linearization_weights * conv_outputs, axis=-1)
    cam = tf.nn.relu(cam)

    heatmap = cam / (tf.reduce_max(cam) + 1e-7)
    heatmap = tf.image.resize(heatmap[..., tf.newaxis], (img_size, img_size))
    heatmap = tf.squeeze(heatmap).numpy()
    return heatmap

# find last Conv2D layer
last_conv_name = None
for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_name = layer.name
        break
print("Using last conv layer for Grad-CAM++:", last_conv_name)

# ------------------------------------------------------------
# 11) Select one test image and run predictions & Grad-CAM++
# ------------------------------------------------------------
idx = 0   # change this index to see different test images
img_path = test_gen.filepaths[idx]
print("Explaining image:", img_path)

orig_bgr = cv2.imread(img_path)
orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
orig_rgb_resized = cv2.resize(orig_rgb, (IMG_SIZE, IMG_SIZE))

# preprocess for DenseNet
inp = densenet_preprocess(orig_rgb_resized.astype(np.float32))
inp_batch = np.expand_dims(inp, axis=0)

# prediction
pred_prob = float(model.predict(inp_batch)[0, 0])
pred_label = "PNEUMONIA" if pred_prob >= 0.5 else "NORMAL"

print(f"Predicted probability (pneumonia) = {pred_prob:.4f}")
print("Final prediction:", pred_label)

# Grad-CAM++ heatmap
heatmap_pp = make_gradcampp_heatmap(inp_batch, model, last_conv_name, img_size=IMG_SIZE)
heatmap_uint8 = np.uint8(255 * heatmap_pp)
heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

overlay = cv2.addWeighted(
    orig_rgb_resized.astype(np.float32) / 255.0,
    0.6,
    heatmap_color.astype(np.float32) / 255.0,
    0.4,
    0.0
)
overlay_uint8 = np.uint8(255 * overlay)

plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.imshow(orig_rgb_resized); plt.title("Original"); plt.axis('off')
plt.subplot(1,3,2); plt.imshow(heatmap_pp, cmap='jet'); plt.title("Grad-CAM++ heatmap"); plt.axis('off')
plt.subplot(1,3,3); plt.imshow(overlay_uint8); plt.title(f"Overlay ({pred_label}, p={pred_prob:.3f})"); plt.axis('off')
plt.tight_layout()
plt.show()

print("Grad-CAM++ visualization done.")
